# Stage 1: Data Acquisition

Loads the raw SVI, PLACES, and NCHS datasets from Google Drive and does an
initial raw-data profiling pass.

**Requires:** raw CSVs in the Drive `data` folder.
**Produces:** nothing persisted — Stage 2 (`data_cleaning.ipynb`) reloads the
raw CSVs directly since re-reading them is cheap.

In [10]:
# =====================================================================
# PMOS UNDERDIAGNOSIS RISK PROJECT
# AI4ALL Group 14B
# Group Names: Irene Zhang, Joyce Xu, Walter Valera,
# Vaishali Allibada, Andy Romero

# Title: Predicting County-Level PMOS Underdiagnosis Risk Across the US
# Datasets:
#   CDC Social Vulnerability Index, 2022 US Counties
#   CDC PLACES: Local Data for Better Health, County Data 2025 release
#   NCHS 2023 Urban-Rural Classification Scheme
# ML Models: Random Forest, Logistic Regression, XGBoost
# Label: Binary Outcome (high risk (1) vs low risk (0) county)
# Literature: Silva et al. 2024, Ramphul et al. 2025, Neven et al. 2026
# =====================================================================

In [11]:
# IMPORT LIBRARIES
import os # file path operations
import pandas as pd # data loading/manipulation

from google.colab import drive

print("✅ SUCCESS: All libraries imported")

✅ SUCCESS: All libraries imported


In [13]:
# MOUNTING DRIVE AND LOADING RAW DATASETS - connect Colab to Drive folder
# Raw data files untouched, cleaning occurs on copies

# User permission to connect to personal Google Drive storage
drive.mount("/content/drive")

# [EDIT AS NEEDED] Path to data folder
# TROUBLESHOOTING: Select folder > Organize > Add to Shortcut > MyDrive
FOLDER_NAME = "data"
DATASET_PATH = f"/content/drive/MyDrive/{FOLDER_NAME}"

# Path error check
if os.path.exists(DATASET_PATH):
  print("✅ SUCCESS: Dataset folder located.")
  print("Files available:", os.listdir(DATASET_PATH))

  # Name files
  svi_file = "SVI_2022_US_county.csv"
  places_file = "PLACES__Local_Data_for_Better_Health,_County_Data,_2025_release_20260725.csv"
  nchs_file = "NCHSurb-rural-codes.csv"

  # Load data

  # SVI 2022 - 3144 rows (/county), 158 cols
  df_svi_raw = pd.read_csv(f"{DATASET_PATH}/{svi_file}")

  # PLACES 2025 - ~240k rows (/county x health measure)
  df_places_raw = pd.read_csv(f"{DATASET_PATH}/{places_file}",
                              low_memory=False) # account for file size

  # NCHS: one row/county with 1-6 NCHS code
  df_nchs_raw = pd.read_csv(f"{DATASET_PATH}/{nchs_file}",
                            encoding="latin-1") # acc for special characters

  print ("✅ SUCCESS: All data sets loaded.")
  print(f"SVI:    {df_svi_raw.shape[0]} counties, {df_svi_raw.shape[1]} columns")
  print(f"PLACES: {df_places_raw.shape[0]} rows, {df_places_raw.shape[1]} columns")
  print(f"NCHS:   {df_nchs_raw.shape[0]} rows, {df_nchs_raw.shape[1]} columns")

else:
  print(f"❌ ERROR: {FOLDER_NAME} was not found in your MyDrive.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ SUCCESS: Dataset folder located.
Files available: ['PLACES__Local_Data_for_Better_Health,_County_Data,_2025_release_20260725.csv', 'NCHSurb-rural-codes.csv', 'SVI_2022_US_county.csv']
✅ SUCCESS: All data sets loaded.
SVI:    3144 counties, 158 columns
PLACES: 229298 rows, 22 columns
NCHS:   3160 rows, 11 columns


In [14]:
# ANALYZING RAW DATA - run once to visualize before cleaning
# Key factors:
# Column titles (selecting features/joining datasets)
# -999 SVI values = missing/unavail census data; not used after
# RPL scores range 0-1 | EP_ cols range 0-100
# PLACES MeasureID strings (filter measures)
# NCHS rural code col name/scale (1-6)

print("==== SVI: column titles first 5 rows ====")
print(df_svi_raw.columns.tolist())
print(df_svi_raw.head(5))

print("\n==== SVI: RPL column stats ====")
rpl_cols = [c for c in df_svi_raw.columns if c.startswith("RPL_")]
print(df_svi_raw[rpl_cols].describe().round(3)) # look for -999 as min

print("\n==== SVI: # missing values (top 10 columns) ====")
print(df_svi_raw.isna().sum().sort_values(ascending=False).head(10))

print("\n==== PLACES: column titles and first 5 rows ====")
print(df_places_raw.columns.tolist())
print(df_places_raw.head(5))

print("\n==== PLACES: unique MeasureIDS ====") # find strings here
print(sorted(df_places_raw["MeasureId"].unique()))

print("\n==== NCHS: column titles and first 5 rows =====")
print(df_nchs_raw.columns.tolist())
print(df_nchs_raw.head(5))

print("\n==== FIPS FORMAT CHECK =====")
print("SVI FIPS: ", df_svi_raw["FIPS"].head(3).tolist())
print("PLACES FIPS: ", df_places_raw["LocationID"].head(3).tolist())

==== SVI: column titles first 5 rows ====
['ST', 'STATE', 'ST_ABBR', 'STCNTY', 'COUNTY', 'FIPS', 'LOCATION', 'AREA_SQMI', 'E_TOTPOP', 'M_TOTPOP', 'E_HU', 'M_HU', 'E_HH', 'M_HH', 'E_POV150', 'M_POV150', 'E_UNEMP', 'M_UNEMP', 'E_HBURD', 'M_HBURD', 'E_NOHSDP', 'M_NOHSDP', 'E_UNINSUR', 'M_UNINSUR', 'E_AGE65', 'M_AGE65', 'E_AGE17', 'M_AGE17', 'E_DISABL', 'M_DISABL', 'E_SNGPNT', 'M_SNGPNT', 'E_LIMENG', 'M_LIMENG', 'E_MINRTY', 'M_MINRTY', 'E_MUNIT', 'M_MUNIT', 'E_MOBILE', 'M_MOBILE', 'E_CROWD', 'M_CROWD', 'E_NOVEH', 'M_NOVEH', 'E_GROUPQ', 'M_GROUPQ', 'EP_POV150', 'MP_POV150', 'EP_UNEMP', 'MP_UNEMP', 'EP_HBURD', 'MP_HBURD', 'EP_NOHSDP', 'MP_NOHSDP', 'EP_UNINSUR', 'MP_UNINSUR', 'EP_AGE65', 'MP_AGE65', 'EP_AGE17', 'MP_AGE17', 'EP_DISABL', 'MP_DISABL', 'EP_SNGPNT', 'MP_SNGPNT', 'EP_LIMENG', 'MP_LIMENG', 'EP_MINRTY', 'MP_MINRTY', 'EP_MUNIT', 'MP_MUNIT', 'EP_MOBILE', 'MP_MOBILE', 'EP_CROWD', 'MP_CROWD', 'EP_NOVEH', 'MP_NOVEH', 'EP_GROUPQ', 'MP_GROUPQ', 'EPL_POV150', 'EPL_UNEMP', 'EPL_HBURD', 'EPL_N